# Day 4 — Orchestration & Quality Practice Questions
## Topic: PySpark DQ checks · Watermark · Upsert · Business SQL patterns

---

> **Rules:**
> - All bronze/silver/gold tables must exist (run Days 1–3 first)
> - Q1 and Q2 use PySpark; Q3 is a business SQL challenge

| # | Difficulty | Topic |
|---|-----------|-------|
| Q1 | 🟢 Easy | PySpark DQ: outlier check + left_anti referential integrity |
| Q2 | 🟡 Medium | PySpark incremental load with watermark and dedup |
| Q3 | 🔴 Hard | SQL: MoM revenue growth with LAG + CASE WHEN trend label |

---
## Setup — run once

In [ ]:
import sys, os
from datetime import date, datetime
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from config.db_config import (
    get_engine, set_spark_env, get_spark, new_batch,
    spark_read_pg, spark_write_pg, JDBC_URL, DB_USER, DB_PASS
)
from sqlalchemy import text

engine = get_engine()
BATCH_ID, INGESTED_AT = new_batch()

set_spark_env()
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType

spark = get_spark('Day4_PQ')

def read_silver_spark(table):
    return spark_read_pg(spark, 'silver', table)

print('Setup ready.')

---
---
## Q1 — 🟢 Easy
### PySpark DQ: statistical outlier check + referential integrity

---

### Problem Statement

Part A — Outlier check:
1. Read `silver.orders` → PySpark DataFrame, cast `total_amount` to `DoubleType()`
2. Filter to non-cancelled orders using `.filter(F.col('is_cancelled') == False)`
3. Compute mean and std using `.agg(F.mean(), F.stddev())`; extract with `.collect()[0]`
4. Flag outliers: rows where `total_amount > mean + 3 * std`
5. Print mean, std, threshold, outlier count
6. PASS if 0 outliers, FAIL otherwise

Part B — Referential integrity using `left_anti` join:
7. Read `silver.order_items` and `silver.products` → PySpark
8. Find `order_items` rows with no matching `product_id` in `products` using `.join(..., how='left_anti')`
9. PASS if 0 orphans, FAIL otherwise

**Hints:** `df.agg(F.mean('col').alias('m'), F.stddev('col').alias('s')).collect()[0]`, `.join(df2.select('pk'), on='pk', how='left_anti').count()`

In [ ]:
# Q1 — Write your solution here


<details><summary>💡 Solution</summary>

```python
dq = []

# Part A: outlier check
df_ord = read_silver_spark('orders')
df_ord = df_ord.withColumn('total_amount', F.col('total_amount').cast(DoubleType()))
df_active = df_ord.filter(F.col('is_cancelled') == False)

stats = df_active.agg(
    F.mean('total_amount').alias('m'),
    F.stddev('total_amount').alias('s')
).collect()[0]

mean, std = stats['m'], stats['s']
threshold = mean + 3 * std
n_outliers = df_active.filter(F.col('total_amount') > threshold).count()

print(f'Mean: {mean:.2f}  Std: {std:.2f}  Threshold: {threshold:.2f}')
print(f'Outliers > threshold: {n_outliers}')

dq.append({'Check': 'orders: no 3σ outliers',
           'Status': 'PASS' if n_outliers == 0 else 'FAIL',
           'Detail': f'{n_outliers} outliers above {threshold:.2f}'})

# Part B: referential integrity
df_items = read_silver_spark('order_items')
df_prod  = read_silver_spark('products')

orphans = df_items.join(df_prod.select('product_id'), on='product_id', how='left_anti').count()
dq.append({'Check': 'order_items.product_id → products',
           'Status': 'PASS' if orphans == 0 else 'FAIL',
           'Detail': f'{orphans} orphans'})

print()
print(f'  {"Check":<40} {"Status":<6} Detail')
print('-' * 60)
for r in dq:
    mark = '✓' if r['Status'] == 'PASS' else '✗'
    print(f'  [{mark}] {r["Check"]:<38} {r["Status"]:<6} {r["Detail"]}')
```
</details>

---
---
## Q2 — 🟡 Medium
### PySpark incremental load with watermark + dedup filter

---

### Problem Statement

Simulate an incremental batch where some `order_id` values already exist in bronze:

1. Read `MAX(order_date)` from `bronze.orders` as the watermark using `engine.connect()` + `text()`
2. Create a batch as a Python list of dicts:
   ```
   O050 — existing order_id (should be skipped by dedup)
   O060 — new, today's date
   O061 — new, today's date
   ```
3. Convert batch to a **PySpark** DataFrame using `spark.createDataFrame()`
4. Filter batch to rows where `order_date > watermark` using `F.col() > F.lit(str(wm))`
5. Remove rows whose `order_id` already exists in bronze:
   - Read existing IDs via JDBC: `spark_read_pg(spark, 'bronze', 'orders').select('order_id').distinct()`
   - Use `.join(existing_ids, on='order_id', how='left_anti')`
6. Print: original batch size, after watermark filter, after dedup
7. Append the clean rows to `bronze.orders` using `spark_write_pg(df, 'bronze', 'orders', mode='append')`

**Hints:** `F.lit(str(watermark))`, `how='left_anti'` returns rows with NO match in right side

In [ ]:
# Q2 — Write your solution here


<details><summary>💡 Solution</summary>

```python
today = str(date.today())

# Step 1: watermark
with engine.connect() as conn:
    wm = conn.execute(text('SELECT MAX(order_date) FROM bronze.orders')).scalar()
print(f'Watermark: {wm}')

# Step 2: batch as PySpark DataFrame
batch_data = [
    {'order_id': 'O050', 'customer_id': 'C009', 'order_date': today, 'status': 'pending',
     'payment_method': 'paypal',       'total_amount': 79.99,  'shipping_city': 'Dallas'},
    {'order_id': 'O060', 'customer_id': 'C001', 'order_date': today, 'status': 'pending',
     'payment_method': 'credit_card',  'total_amount': 149.99, 'shipping_city': 'New York'},
    {'order_id': 'O061', 'customer_id': 'C005', 'order_date': today, 'status': 'shipped',
     'payment_method': 'debit_card',   'total_amount': 89.50,  'shipping_city': 'Phoenix'},
]
df_batch = spark.createDataFrame(batch_data)
print(f'Batch size         : {df_batch.count()}')

# Step 3: watermark filter
df_wm = df_batch.filter(F.col('order_date') > F.lit(str(wm)))
print(f'After watermark    : {df_wm.count()}')

# Step 4: dedup via left_anti join on existing bronze IDs (pure PySpark JDBC)
existing_ids = spark_read_pg(spark, 'bronze', 'orders').select('order_id').distinct()
df_clean = df_wm.join(existing_ids, on='order_id', how='left_anti')
print(f'After dedup filter : {df_clean.count()}')

# Step 5: append to bronze via JDBC
if df_clean.count() > 0:
    df_meta = (df_clean
        .withColumn('_source_file', F.lit('pq_incremental_batch'))
        .withColumn('_ingested_at', F.lit(INGESTED_AT))
        .withColumn('_batch_id',    F.lit(BATCH_ID))
    )
    spark_write_pg(df_meta, 'bronze', 'orders', mode='append')

with engine.connect() as conn:
    total = conn.execute(text('SELECT COUNT(*) FROM bronze.orders')).scalar()
print(f'bronze.orders total: {total}')
```
</details>

---
---
## Q3 — 🔴 Hard
### SQL: Month-over-month revenue growth with trend label

---

### Problem Statement

Write a single SQL query (using CTEs) that:

1. Computes monthly revenue from `silver.orders` (non-cancelled)
2. Uses `LAG()` to get previous month's revenue
3. Computes `growth_pct = 100 * (current - prev) / prev` (rounded to 1dp, null for first month)
4. Adds a `trend` label:
   - `'↑ growth'` if growth_pct > 0
   - `'↓ decline'` if growth_pct < 0
   - `'→ flat'` if growth_pct = 0
   - null if no previous month
5. Return: `month`, `revenue`, `prev_revenue`, `growth_pct`, `trend`
6. Order by month ascending

**Hints:** `LAG(col) OVER (ORDER BY month)`, `CASE WHEN growth_pct > 0 THEN ...`

In [ ]:
# Q3 — Write your SQL here, then create a view and read via PySpark
sql = """
-- Your SQL here (wrap it in a view)
"""

# How to run it:
# with engine.connect() as conn:
#     conn.execute(text('CREATE OR REPLACE VIEW gold.v_pq_mom_growth AS ' + sql))
#     conn.commit()
# spark_read_pg(spark, 'gold', 'v_pq_mom_growth').show(truncate=False)

<details><summary>💡 Solution</summary>

```python
inner_sql = """
    WITH monthly AS (
        SELECT
            DATE_TRUNC('month', order_date)        AS order_month,
            ROUND(SUM(total_amount)::NUMERIC, 2)   AS revenue
        FROM silver.orders
        WHERE is_cancelled = FALSE
        GROUP BY DATE_TRUNC('month', order_date)
    ),
    with_lag AS (
        SELECT
            TO_CHAR(order_month, 'YYYY-MM') AS month,
            revenue,
            ROUND(LAG(revenue) OVER (ORDER BY order_month)::NUMERIC, 2) AS prev_revenue,
            CASE
                WHEN LAG(revenue) OVER (ORDER BY order_month) IS NOT NULL
                THEN ROUND(
                    100.0 * (revenue - LAG(revenue) OVER (ORDER BY order_month))
                    / LAG(revenue) OVER (ORDER BY order_month), 1
                )
            END AS growth_pct
        FROM monthly
    )
    SELECT
        month, revenue, prev_revenue, growth_pct,
        CASE
            WHEN growth_pct IS NULL THEN NULL
            WHEN growth_pct > 0     THEN '↑ growth'
            WHEN growth_pct < 0     THEN '↓ decline'
            ELSE                         '→ flat'
        END AS trend
    FROM with_lag
    ORDER BY month
"""

with engine.connect() as conn:
    conn.execute(text(f'CREATE OR REPLACE VIEW gold.v_pq_mom_growth AS {inner_sql}'))
    conn.commit()

spark_read_pg(spark, 'gold', 'v_pq_mom_growth').show(truncate=False)
```
</details>

In [ ]:
spark.stop()
print('Spark stopped.')